        # Discusión de resultados en función del problema de estudio

        **Modelación y Simulación Computacional** · Maestría en Ingeniería ·
        Universidad de Sucre · periodo 2026-2

        **Unidad 3.** Simulación de sistemas y análisis de escenarios ·
        **Subtema del plan 3.5**

        Autor, Prof. Daniel Otero Meza, Ing., Ph.D.

        <!-- ENLACE_COLAB -->
        [![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://github.com/<usuario>/<repositorio>/blob/main/03_cuadernos/Unidad3/U3_05_discusion_de_resultados.ipynb)

        El paso final es el que más se descuida y consiste en discutir los
resultados en función del problema de estudio y no del modelo. Una tabla
de corridas se convierte en recomendación cuando se declara qué
alternativas quedaron descartadas y por qué, cuál es el precio de cada
mejora sobre la frontera, qué tan sensible es el orden de preferencia al
cambio de contexto y qué información faltante impide decidir. Este
cuaderno reúne los resultados de los cuatro anteriores y los convierte en
un informe defendible.

        ## Objetivos de aprendizaje

        Al terminar este cuaderno el estudiante debe ser capaz de

        1. Leer sobre la frontera de compromiso el precio de cada mejora y expresarlo en las unidades del problema.
2. Evaluar la robustez de una recomendación al cambio de contexto y distinguirla de la mejor alternativa en un contexto único.
3. Redactar el conjunto mínimo defendible de cifras que debe acompañar a una simulación estocástica.
4. Justificar con cifras por qué el valor medio no es la magnitud de diseño cuando el suministro debe garantizarse.
5. Consolidar en una sola ficha todas las verificaciones de la unidad frente a las cifras que publica el capítulo.

## Puesta a punto

La primera celda detecta el entorno e instala solo lo que falte, de modo que
el cuaderno abre igual en Google Colab y en JupyterLab. La segunda fija la
semilla del curso, la paleta del libro y las funciones auxiliares. La semilla
vale 20262 y ningún resultado depende de una ejecución concreta.

In [ ]:
import importlib
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules


def asegurar(paquetes: dict) -> None:
    """Instala solo los paquetes que no estén disponibles."""
    faltantes = [p for p, m in paquetes.items()
                 if importlib.util.find_spec(m) is None]
    if faltantes:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltantes],
                       check=True)


asegurar({"numpy": "numpy", "scipy": "scipy", "pandas": "pandas",
          "matplotlib": "matplotlib", "sympy": "sympy"})
print("entorno listo, Colab =", EN_COLAB)

In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEMILLA = 20262
rng = np.random.default_rng(SEMILLA)

COLORES = {"azul": "#1F4E79", "rojo": "#B3251E", "verde": "#2E7D32",
           "naranja": "#E07B00", "gris": "#5A5A5A", "morado": "#6A3D9A"}

plt.rcParams.update({"figure.figsize": (9.0, 4.4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 10, "legend.frameon": False})

trapecio = np.trapezoid if hasattr(np, "trapezoid") else np.trapz

pd.set_option("display.width", 110)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")


def carpeta_datos() -> Path:
    """Ubica la carpeta datos sin usar rutas absolutas.

    Busca hacia arriba desde el directorio de trabajo, de modo que funcione
    tanto en el repositorio como en una sesión de Colab donde el cuaderno se
    abre suelto. Si no la encuentra, la crea junto al cuaderno.
    """
    base = Path.cwd()
    for candidata in (base, *list(base.parents)[:3]):
        if (candidata / "datos").is_dir():
            return candidata / "datos"
    destino = base / "datos"
    destino.mkdir(exist_ok=True)
    return destino


def leer_datos(nombre: str, respaldo) -> pd.DataFrame:
    """Lee un archivo de datos y lo reconstruye si no está disponible.

    El argumento respaldo es una función sin argumentos que devuelve el
    mismo cuadro de datos, construido con las cifras publicadas en el libro.
    Así el cuaderno nunca depende de una descarga.
    """
    base = Path.cwd()
    for candidata in (base, *list(base.parents)[:3]):
        ruta = candidata / "datos" / nombre
        if ruta.exists():
            return pd.read_csv(ruta)
    tabla = respaldo()
    tabla.to_csv(carpeta_datos() / nombre, index=False)
    return tabla


def comparar(etiqueta: str, calculado: float, libro: float,
             tol: float, unidad: str = "") -> bool:
    """Imprime y verifica un valor calculado frente al que publica el libro."""
    dif = abs(calculado - libro)
    ok = dif <= tol
    marca = "coincide" if ok else "NO coincide"
    print(f"{etiqueta:<46s} calculado {calculado:>14.6g} {unidad:<12s}"
          f" libro {libro:>12.6g}   {marca}")
    return ok


print("semilla del curso", SEMILLA)

In [ ]:
def respaldo_valores_libro() -> pd.DataFrame:
    """Cifras publicadas en el capítulo 3, transcritas del libro."""
    filas = [
    ("colebrook_velocidad", 1.6977, "m/s", "Ejemplo 3.1"),
    ("colebrook_reynolds", 507267.0, "adimensional", "Ejemplo 3.1"),
    ("colebrook_rugosidad_relativa", 0.0008667, "adimensional", "Ejemplo 3.1"),
    ("colebrook_factor_friccion", 0.0196228, "adimensional", "Ejemplo 3.1"),
    ("colebrook_perdida_carga", 8.167, "m", "Ejemplo 3.1"),
    ("colebrook_swamee_jain", 0.019742, "adimensional", "Ejemplo 3.1"),
    ("colebrook_orden_newton", 2.0, "adimensional", "Ejemplo 3.1"),
    ("lagunas_perfil_1", 142.42, "mg/L", "seccion 3.1.2"),
    ("lagunas_perfil_2", 83.4, "mg/L", "seccion 3.1.2"),
    ("lagunas_perfil_3", 41.5, "mg/L", "seccion 3.1.2"),
    ("lagunas_perfil_4", 17.65, "mg/L", "seccion 3.1.2"),
    ("lagunas_perfil_5", 7.33, "mg/L", "seccion 3.1.2"),
    ("lagunas_remocion", 97.07, "por ciento", "seccion 3.1.2"),
    ("lagunas_retencion", 8.33, "d", "seccion 3.1.2"),
    ("lagunas_carga_afluente", 300000.0, "mg/d", "seccion 3.1.2"),
    ("lagunas_carga_efluente", 8792.84, "mg/d", "seccion 3.1.2"),
    ("lagunas_consumo", 291207.16, "mg/d", "seccion 3.1.2"),
    ("fermentador_tiempo_25C", 17.0604614, "h", "Ejemplo 3.2"),
    ("fermentador_tiempo_30C", 10.9186953, "h", "Ejemplo 3.2"),
    ("fermentador_tiempo_35C", 7.5824273, "h", "Ejemplo 3.2"),
    ("fermentador_invariante", 12.5, "g/L", "Ejemplo 3.2"),
    ("fermentador_mumax_25C", 0.192, "1/h", "Ejemplo 3.2"),
    ("fermentador_mumax_30C", 0.3, "1/h", "Ejemplo 3.2"),
    ("fermentador_mumax_35C", 0.432, "1/h", "Ejemplo 3.2"),
    ("fermentador_evaluaciones", 584.0, "evaluaciones", "Ejemplo 3.2"),
    ("tolerancia_tiempo_rtol3", 10.9028, "h", "seccion 3.2.1"),
    ("tolerancia_error_rtol3", 0.00146, "adimensional", "seccion 3.2.1"),
    ("tolerancia_evaluaciones_rtol3", 44.0, "evaluaciones", "seccion 3.2.1"),
    ("tolerancia_error_rtol6", 1.85e-07, "adimensional", "seccion 3.2.1"),
    ("tolerancia_evaluaciones_rtol6", 194.0, "evaluaciones", "seccion 3.2.1"),
    ("tolerancia_error_rtol9", 2.45e-10, "adimensional", "seccion 3.2.1"),
    ("tolerancia_evaluaciones_rtol9", 584.0, "evaluaciones", "seccion 3.2.1"),
    ("circuito_autovalor_rapido", -1005.0002, "1/s", "Ejemplo 3.3"),
    ("circuito_autovalor_lento", -0.0497512, "1/s", "Ejemplo 3.3"),
    ("circuito_razon_rigidez", 20200.0, "adimensional", "Ejemplo 3.3"),
    ("circuito_pasos_rk45", 30376.0, "pasos", "Ejemplo 3.3"),
    ("circuito_evaluaciones_rk45", 212576.0, "evaluaciones", "Ejemplo 3.3"),
    ("circuito_pasos_bdf", 144.0, "pasos", "Ejemplo 3.3"),
    ("circuito_evaluaciones_bdf", 292.0, "evaluaciones", "Ejemplo 3.3"),
    ("circuito_paso_medio_rk45", 0.003292, "s", "Ejemplo 3.3"),
    ("circuito_tau_rapida", 0.000995, "s", "Ejemplo 3.3"),
    ("circuito_tau_lenta", 20.1, "s", "Ejemplo 3.3"),
    ("circuito_error_rk45", 9.1e-07, "adimensional", "Ejemplo 3.3"),
    ("circuito_error_bdf", 1.1e-06, "adimensional", "Ejemplo 3.3"),
    ("circuito_producto_h_lambda", 3.31, "adimensional", "Ejemplo 3.3"),
    ("rio_peclet_celda", 0.583, "adimensional", "Ejemplo 3.4"),
    ("rio_pico_analitico", 1.8655, "mg/L", "Ejemplo 3.4"),
    ("rio_abscisa_pico", 2460.0, "m", "Ejemplo 3.4"),
    ("rio_error_maximo", 7.18e-06, "kg/m3", "Ejemplo 3.4"),
    ("rio_masa_remanente", 24.740935, "kg", "Ejemplo 3.4"),
    ("rio_orden_observado", 2.0, "adimensional", "Ejemplo 3.4"),
    ("rio_paso_difusion", 16.67, "s", "seccion 3.3.2"),
    ("rio_paso_adveccion", 57.14, "s", "seccion 3.3.2"),
    ("rio_error_explicito_d045", 6.3e-05, "kg/m3", "Ejemplo 3.4"),
    ("riego_frontera_bruto_p045_d45", 393.8, "mm", "Ejemplo 3.5"),
    ("riego_frontera_bruto_p085_d65", 243.8, "mm", "Ejemplo 3.5"),
    ("riego_deficit_p085_d65", 21.5, "por ciento", "Ejemplo 3.5"),
    ("riego_no_dominadas_clima_normal", 5.0, "alternativas", "Ejemplo 3.5"),
    ("biogas_desviacion_replicas_mc", 0.933, "kW h/d", "Ejemplo 3.6"),
    ("biogas_desviacion_replicas_lhs", 0.112, "kW h/d", "Ejemplo 3.6"),
    ("biogas_reduccion_varianza", 69.0, "veces", "Ejemplo 3.6"),
    ("riego_agua_aprovechable", 126.0, "mm", "Ejemplo 3.5"),
    ("biogas_media", 92.34, "kW h/d", "Ejemplo 3.6"),
    ("biogas_desviacion", 21.22, "kW h/d", "Ejemplo 3.6"),
    ("biogas_p05", 61.71, "kW h/d", "Ejemplo 3.6"),
    ("biogas_p50", 90.05, "kW h/d", "Ejemplo 3.6"),
    ("biogas_p95", 130.62, "kW h/d", "Ejemplo 3.6"),
    ("biogas_excedencia_110", 0.1945, "adimensional", "Ejemplo 3.6"),
    ("biogas_error_estandar", 0.15, "kW h/d", "Ejemplo 3.6"),
    ("biogas_valores_centrales", 91.53, "kW h/d", "Ejemplo 3.6"),
    ("lcoe_crf", 0.101806, "1/a", "Ejemplo 3.7"),
    ("lcoe_factor_degradacion", 0.931205, "adimensional", "Ejemplo 3.7"),
    ("lcoe_produccion_especifica", 1325.6, "kW h/(kW a)", "Ejemplo 3.7"),
    ("lcoe_nominal", 0.083523, "USD/(kW h)", "Ejemplo 3.7"),
    ("lcoe_elasticidad_inversion", 0.87355, "adimensional", "Ejemplo 3.7"),
    ("lcoe_elasticidad_tasa", 0.637, "adimensional", "Ejemplo 3.7"),
    ("lcoe_amplitud_tasa", 35.8, "por ciento", "Ejemplo 3.7"),
    ("lcoe_amplitud_irradiacion", 16.1, "por ciento", "Ejemplo 3.7"),
    ("ishigami_s1", 0.3138, "adimensional", "seccion 3.6.2"),
    ("ishigami_s2", 0.4423, "adimensional", "seccion 3.6.2"),
    ("ishigami_s3", -0.0001, "adimensional", "seccion 3.6.2"),
    ("ishigami_st3", 0.2436, "adimensional", "seccion 3.6.2"),
    ("sobol_biogas_s_B0", 0.616, "adimensional", "seccion 3.6.2"),
    ("sobol_biogas_suma_primer_orden", 0.985, "adimensional", "seccion 3.6.2"),
    ("morris_evaluaciones", 210.0, "evaluaciones", "seccion 3.6.2"),
    ]
    return pd.DataFrame(filas, columns=["clave", "valor", "unidad", "referencia"])


LIBRO = leer_datos("valores_libro_cap3.csv",
                   respaldo_valores_libro).set_index("clave")["valor"]
print(f"cifras del libro disponibles, {LIBRO.size} registros")

## 1. De la tabla de corridas a la recomendación

Se reconstruye el barrido de escenarios del Ejemplo 3.5 con el mismo
balance hídrico del cuaderno tercero, que reproduce de manera exacta las
veintisiete láminas brutas de la Tabla 3.3. La curva de coeficiente de
cultivo se lee de `datos/kc_maiz_90d.csv`.

In [ ]:
import itertools

THETA_CC, THETA_PM, PROFUNDIDAD = 0.29, 0.15, 0.90
FRACCION_SIN_ESTRES, EFICIENCIA = 0.55, 0.80
TAW = 1000 * (THETA_CC - THETA_PM) * PROFUNDIDAD
RAW = FRACCION_SIN_ESTRES * TAW


def respaldo_kc() -> pd.DataFrame:
    """Curva trapezoidal de coeficiente de cultivo para 90 días."""
    largo_ini, largo_des, largo_med, largo_fin = 12, 37, 34, 7
    kc_ini, kc_med, kc_fin = 0.35, 1.15, 0.40
    curva = [kc_ini] * largo_ini
    curva += [kc_ini + (kc_med - kc_ini) * j / largo_des
              for j in range(1, largo_des + 1)]
    curva += [kc_med] * largo_med
    curva += [kc_med + (kc_fin - kc_med) * j / largo_fin
              for j in range(1, largo_fin + 1)]
    return pd.DataFrame({"dia": np.arange(1, 91), "kc": np.round(curva, 6)})


KC = leer_datos("kc_maiz_90d.csv", respaldo_kc)["kc"].to_numpy()


def balance_hidrico(p: float, d: float, ETo: float) -> dict:
    """Balance hídrico diario del suelo, con láminas en milímetros."""
    agotamiento, riegos = 0.0, 0
    transpiracion, potencial, percolacion = 0.0, 0.0, 0.0
    for kc_dia in KC:
        estres = min(1.0, max(0.0, (TAW - agotamiento) / (TAW - RAW)))
        etc = kc_dia * ETo
        eta = estres * etc
        potencial += etc
        transpiracion += eta
        agotamiento += eta
        if agotamiento >= p * TAW:
            agotamiento -= d
            riegos += 1
        if agotamiento < 0.0:
            percolacion -= agotamiento
            agotamiento = 0.0
        agotamiento = min(agotamiento, TAW)
    return dict(riegos=riegos, bruto=riegos * d / EFICIENCIA,
                deficit=100 * (1 - transpiracion / potencial),
                percolacion=percolacion, agotamiento=agotamiento,
                transpiracion=transpiracion)


NIVELES = {"p": [0.45, 0.65, 0.85], "d": [25.0, 45.0, 65.0],
           "ETo": [4.0, 4.8, 5.6]}
barrido = pd.DataFrame([
    {**dict(zip(NIVELES, c)), **balance_hidrico(*c)}
    for c in itertools.product(*NIVELES.values())])
barrido["bruto"] = np.round(barrido["bruto"], 1)


def frontera_compromiso(tabla, columnas=("bruto", "deficit")):
    """Alternativas no dominadas, con ambos indicadores a minimizar."""
    a, b = columnas
    dominado = tabla.apply(
        lambda fila: bool(((tabla[a] <= fila[a]) & (tabla[b] <= fila[b])
                           & ((tabla[a] < fila[a]) | (tabla[b] < fila[b]))
                           ).any()), axis=1)
    return tabla[~dominado].sort_values(a)


normal = barrido.query("ETo == 4.8").reset_index(drop=True)
frente = frontera_compromiso(normal)
print(frente[["p", "d", "riegos", "bruto", "deficit"]].to_string(
    index=False, formatters={"bruto": "{:8.1f}".format,
                             "deficit": "{:7.2f}".format}))

### Ejercicio 1

La frontera hace un trabajo doble, pues elimina las alternativas
dominadas, sobre las cuales no cabe discusión razonable, y expone el
precio de cada mejora sobre las que quedan. Complete la función que
recorre la frontera ordenada por lámina bruta y devuelve, para cada
paso, el agua adicional que cuesta y los puntos de déficit que evita. La
celda de partida devuelve ceros.

In [ ]:
# COMPLETE: para cada par de alternativas contiguas de la frontera,
#   agua adicional = bruto siguiente menos bruto actual
#   déficit evitado = deficit actual menos deficit siguiente
#   precio = agua adicional dividida por el déficit evitado
REVISAR_PRECIO = False


def precio_de_la_mejora(frontera: pd.DataFrame) -> pd.DataFrame:
    """Costo en agua de cada mejora sucesiva sobre la frontera."""
    orden = frontera.sort_values("bruto").reset_index(drop=True)
    filas = []
    for j in range(len(orden) - 1):
        actual, siguiente = orden.loc[j], orden.loc[j + 1]
        filas.append((f"p={actual['p']:.2f} d={actual['d']:.0f}",
                      f"p={siguiente['p']:.2f} d={siguiente['d']:.0f}",
                      0.0, 0.0, 0.0))
    return pd.DataFrame(filas, columns=["desde", "hasta",
                                        "agua adicional (mm)",
                                        "déficit evitado (puntos)",
                                        "precio (mm por punto)"])

In [ ]:
precios = precio_de_la_mejora(frente)
print(precios.to_string(index=False, float_format=lambda v: f"{v:10.2f}"))

salto_grande = precios.iloc[2]["agua adicional (mm)"]
salto_final = precios.iloc[3]["agua adicional (mm)"]
print(f"\npasar de un déficit de {frente['deficit'].iloc[2]:.1f} por ciento "
      f"a uno de {frente['deficit'].iloc[3]:.1f} cuesta {salto_grande:.0f} mm")
print(f"eliminar el déficit por completo cuesta otros {salto_final:.0f} mm")
print("el libro reporta esos mismos 100 mm y 50 mm adicionales, con "
      "déficits de 21.5 y 5.1 por ciento")

if REVISAR_PRECIO:
    assert abs(salto_grande - 100.0) < 0.1, \
        "el libro reporta 100 mm adicionales para esa mejora"
    assert abs(salto_final - 50.0) < 0.1, \
        "el libro reporta otros 50 mm para eliminar el déficit"
    print("\nlos dos precios coinciden con la sección 3.4.1 del libro")
else:
    print("\ncomplete la celda anterior y ponga REVISAR_PRECIO = True")

## 2. Robustez de la recomendación al cambio de contexto

Ninguna de las cinco alternativas de la frontera es la respuesta, que
depende del precio del agua y del valor del rendimiento perdido,
información ausente del modelo hidrológico. Lo que sí puede hacer el
análisis es señalar cuáles alternativas se sostienen en los tres climas y
cuáles solo brillan en uno.

### Ejercicio 2

Complete la función que resume cada alternativa de operación por su peor
desempeño en los tres contextos, que es el criterio conservador que
corresponde cuando el contexto no se controla. La celda de partida
devuelve el desempeño del clima normal únicamente.

In [ ]:
# COMPLETE: agrupe por alternativa y tome el máximo del déficit,
# el máximo de la lámina bruta y el máximo del número de riegos.
REVISAR_ROBUSTEZ = False


def peor_caso(tabla: pd.DataFrame) -> pd.DataFrame:
    """Peor desempeño de cada alternativa a través de los contextos."""
    solo_normal = tabla.query("ETo == 4.8")
    return (solo_normal[["p", "d", "bruto", "deficit", "riegos"]]
            .set_index(["p", "d"]).sort_index())

In [ ]:
peores = peor_caso(barrido)
print(peores.to_string(float_format=lambda v: f"{v:9.2f}"))

robustas = peores.query("deficit <= 2.1")
print("\nalternativas con déficit no mayor al 2.1 por ciento en los tres climas")
print(robustas.to_string(float_format=lambda v: f"{v:9.2f}"))

recomendada = (0.65, 65.0)
fila = peores.loc[recomendada]
print(f"\nrecomendación del libro, umbral 0.65 con lámina de 65 mm")
print(f"  déficit máximo en los tres climas   {fila['deficit']:.2f} por ciento")
print(f"  lámina bruta máxima                 {fila['bruto']:.1f} mm")
print(f"  riegos máximos                      {int(fila['riegos'])}")
print("el libro anuncia cinco riegos o menos, cifra que se cumple en los "
      "climas húmedo y normal, mientras que el clima seco exige seis")

if REVISAR_ROBUSTEZ:
    assert recomendada in robustas.index, \
        "la alternativa recomendada debe quedar bajo el 2.1 por ciento"
    assert fila["deficit"] <= 2.1, "el déficit máximo debe respetar el umbral"
    print("\nla recomendación se sostiene en los tres contextos")
else:
    print("\ncomplete la celda anterior y ponga REVISAR_ROBUSTEZ = True")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.6, 4.0))
for eto, color, marca_pt, etiqueta in ((4.0, COLORES["verde"], "^", "húmedo"),
                                       (4.8, COLORES["azul"], "o", "normal"),
                                       (5.6, COLORES["naranja"], "s", "seco")):
    sub = barrido.query("ETo == @eto")
    ax1.plot(sub["bruto"], sub["deficit"], marca_pt, color=color, ms=5.5,
             mfc="none", label=f"clima {etiqueta}")
ax1.plot(frente["bruto"], frente["deficit"], "--", color=COLORES["rojo"],
         lw=1.2, label="frontera, clima normal")
ax1.set_xlabel("lámina bruta aplicada (mm)")
ax1.set_ylabel("déficit de transpiración (por ciento)")
ax1.set_title("Las nueve alternativas en los tres climas")
ax1.legend(fontsize=8.5)

etiquetas = [f"p={a:.2f}\nd={b:.0f}" for a, b in peores.index]
posicion = np.arange(len(peores))
colores_barra = [COLORES["verde"] if v <= 2.1 else COLORES["gris"]
                 for v in peores["deficit"]]
ax2.bar(posicion, peores["deficit"], color=colores_barra)
ax2.axhline(2.1, color=COLORES["rojo"], ls="--", lw=1.0)
ax2.set_xticks(posicion)
ax2.set_xticklabels(etiquetas, fontsize=7.5)
ax2.set_ylabel("peor déficit en los tres climas (por ciento)")
ax2.set_title("Criterio conservador sobre el contexto")
ax2.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

## 3. El reporte mínimo defendible de una simulación estocástica

Un informe que reporta solo el valor esperado transmite una certeza que
la simulación no respalda, y uno que reporta el intervalo sin declarar el
tamaño de muestra ni la semilla no es reproducible. El mínimo defendible
incluye el estimador puntual, su error estándar, los percentiles
relevantes, el número de corridas, el esquema de muestreo y la semilla.
Se retoma el biodigestor del Ejemplo 3.6.

In [ ]:
from scipy.stats import lognorm, norm, triang, uniform

PCI, DEMANDA, N_CORRIDAS = 35.8, 110.0, 20000


def transformar(u: np.ndarray) -> dict:
    """Del hipercubo unitario a las variables físicas del biodigestor."""
    s = np.sqrt(np.log(1 + 0.18**2))
    return dict(m=norm.ppf(u[:, 0], 1200.0, 90.0),
                SV=norm.ppf(u[:, 1], 0.115, 0.012),
                B0=lognorm.ppf(u[:, 2], s, scale=0.21),
                k=uniform.ppf(u[:, 3], 0.10, 0.15),
                TRH=triang.ppf(u[:, 4], 6 / 13, loc=22.0, scale=13.0),
                eta=triang.ppf(u[:, 5], 0.5, loc=0.28, scale=0.08))


def energia(m, SV, B0, k, TRH, eta):
    """Energía eléctrica diaria en kW h por día."""
    return m * SV * B0 * (1 - np.exp(-k * TRH)) * PCI * eta / 3.6


E = energia(**transformar(np.random.default_rng(SEMILLA).random((N_CORRIDAS, 6))))
print(f"muestra de {E.size:,d} realizaciones, media {E.mean():.2f} kW h/d")

### Ejercicio 3

Complete la ficha de reporte. Debe contener el estimador puntual, su
error estándar, la semiamplitud del intervalo al 95 por ciento, los
percentiles cinco, cincuenta y noventa y cinco, la probabilidad de
excedencia de la demanda, el número de corridas, el esquema de muestreo y
la semilla. La celda de partida deja fuera casi todo, que es justamente
el defecto que se quiere evitar.

In [ ]:
# COMPLETE: agregue error estándar, semiamplitud al 95 por ciento,
# percentiles 5, 50 y 95, probabilidad de excedencia, número de
# corridas, esquema de muestreo y semilla.
REVISAR_FICHA = False


def ficha_reporte(muestra: np.ndarray, umbral: float,
                  esquema: str = "aleatorio simple",
                  semilla: int = SEMILLA) -> dict:
    """Conjunto mínimo de cifras que debe acompañar a la simulación."""
    return {"media (kW h/d)": float(muestra.mean())}

In [ ]:
ficha = ficha_reporte(E, DEMANDA)
for clave, valor in ficha.items():
    if isinstance(valor, float):
        print(f"{clave:<42s} {valor:12.4f}")
    else:
        print(f"{clave:<42s} {valor!s:>12s}")

faltantes = {"media (kW h/d)", "error estándar (kW h/d)",
             "percentil 5 (kW h/d)", "número de corridas",
             "esquema de muestreo", "semilla"} - set(ficha)
print(f"\ncampos obligatorios ausentes   {sorted(faltantes) or 'ninguno'}")

if REVISAR_FICHA:
    assert not faltantes, "la ficha debe contener el conjunto mínimo"
    assert abs(ficha["media (kW h/d)"] - LIBRO["biogas_media"]) < 5e-3
    assert abs(ficha["percentil 5 (kW h/d)"] - LIBRO["biogas_p05"]) < 5e-3
    assert abs(ficha["error estándar (kW h/d)"]
               - LIBRO["biogas_error_estandar"]) < 5e-4
    print("la ficha reproduce las cifras del Ejemplo 3.6")
else:
    print("complete la celda anterior y ponga REVISAR_FICHA = True")

## 4. La media no es la magnitud de diseño

La conclusión operativa del Ejemplo 3.6 no es que el biodigestor produzca
92 kW h/d, sino que cubre la demanda supuesta en menos de una quinta
parte de los días. Si el diseño debe garantizar el suministro, la
magnitud pertinente es el percentil bajo y con ella el dimensionamiento
cambia por completo. El problema 3-32 pide exactamente ese argumento con
cifras plausibles.

### Ejercicio 4

Complete la función que devuelve la magnitud de diseño para una cobertura
dada, es decir el valor que la respuesta iguala o supera en esa fracción
de los días, y los días al año en que la demanda quedaría descubierta. La
celda de partida usa la media, que es justamente la práctica que se
quiere criticar.

In [ ]:
# COMPLETE: la magnitud de diseño con cobertura c es el percentil
# de orden 100*(1 - c) de la muestra, los días por debajo de ese
# valor son 365*(1 - c) y el respaldo es la demanda menos el valor
# de diseño cuando resulta positiva.
REVISAR_DISENO = False


def magnitud_de_diseno(muestra: np.ndarray, cobertura: float,
                       demanda: float) -> dict:
    """Valor de diseño, días por debajo y respaldo necesario."""
    return dict(cobertura=cobertura, valor=float(muestra.mean()),
                dias_por_debajo=0.0, respaldo=0.0)

In [ ]:
opciones = pd.DataFrame([magnitud_de_diseno(E, c, DEMANDA)
                         for c in (0.50, 0.80, 0.90, 0.95)])
print(opciones.to_string(index=False, float_format=lambda v: f"{v:10.2f}"))

print(f"\ndimensionar con la media, {E.mean():.1f} kW h/d, deja la demanda "
      f"descubierta {365 * (E < DEMANDA).mean():.0f} días al año")
print(f"dimensionar con el percentil cinco, {np.percentile(E, 5):.1f} kW h/d, "
      "obliga a un respaldo de "
      f"{DEMANDA - np.percentile(E, 5):.1f} kW h/d")

if REVISAR_DISENO:
    assert abs(opciones.loc[3, "valor"] - LIBRO["biogas_p05"]) < 5e-3, \
        "la cobertura del 95 por ciento corresponde al percentil cinco"
    assert 365 * (E >= DEMANDA).mean() < 0.25 * 365, \
        "la demanda se cubre en menos de una cuarta parte de los días"
    print("\nla magnitud de diseño y la media difieren en un factor cercano "
          "a uno y medio, y esa diferencia es la que decide la inversión "
          "en respaldo")
else:
    print("\ncomplete la celda anterior y ponga REVISAR_DISENO = True")

In [ ]:
fig, ax = plt.subplots(figsize=(8.0, 4.2))
orden = np.sort(E)
excedencia = 1 - np.arange(1, orden.size + 1) / orden.size
ax.plot(orden, 100 * excedencia, color=COLORES["azul"], lw=1.3)
ax.axvline(DEMANDA, color=COLORES["verde"], lw=1.2)
ax.axvline(E.mean(), color=COLORES["gris"], ls="--", lw=1.0)
ax.axvline(np.percentile(E, 5), color=COLORES["rojo"], ls="--", lw=1.0)
ax.annotate(f"demanda {DEMANDA:.0f}", (DEMANDA, 82),
            xytext=(6, 0), textcoords="offset points",
            color=COLORES["verde"], fontsize=9)
ax.annotate(f"media {E.mean():.0f}", (E.mean(), 60), xytext=(6, 0),
            textcoords="offset points", color=COLORES["gris"], fontsize=9)
ax.annotate(f"P5 {np.percentile(E, 5):.0f}", (np.percentile(E, 5), 30),
            xytext=(-52, 0), textcoords="offset points",
            color=COLORES["rojo"], fontsize=9)
ax.set_xlabel("energía eléctrica diaria (kW h/d)")
ax.set_ylabel("probabilidad de excedencia (por ciento)")
ax.set_title("Curva de excedencia, lo que la media oculta")
ax.set_xlim(20, 200)
plt.show()

## 5. Qué información falta y qué no puede decir el modelo

La discusión honesta declara los límites. El análisis de escenarios
entrega el conjunto de opciones defendibles y el precio de pasar de una a
otra, que es todo lo que el modelo puede aportar. Elegir entre ellas
exige el precio del agua y el valor del rendimiento perdido, magnitudes
que no están en el balance hídrico. La celda siguiente muestra qué
ocurriría si se introdujeran, y cómo la elección se desplaza con ellas.

In [ ]:
AREA_HA = 1.0
RENDIMIENTO_POTENCIAL = 8.0        # t/ha
SENSIBILIDAD_KY = 1.25             # coeficiente de respuesta del maíz

def costo_total(fila, precio_agua, precio_grano):
    """Costo del agua más el valor del rendimiento perdido, en USD/ha."""
    agua = fila["bruto"] * 10 * precio_agua           # mm por 10 da m3/ha
    perdida = (SENSIBILIDAD_KY * fila["deficit"] / 100
               * RENDIMIENTO_POTENCIAL * precio_grano)
    return agua + perdida


escenarios_precio = [(0.05, 250.0), (0.20, 250.0), (0.50, 250.0),
                     (0.20, 500.0)]
resultados = []
for precio_agua, precio_grano in escenarios_precio:
    costos = normal.apply(costo_total, axis=1, precio_agua=precio_agua,
                          precio_grano=precio_grano)
    elegida = normal.loc[costos.idxmin()]
    resultados.append((precio_agua, precio_grano,
                       f"p={elegida['p']:.2f} d={elegida['d']:.0f}",
                       float(costos.min())))
eleccion = pd.DataFrame(resultados, columns=["precio del agua (USD/m3)",
                                             "precio del grano (USD/t)",
                                             "alternativa elegida",
                                             "costo total (USD/ha)"])
print(eleccion.to_string(index=False, float_format=lambda v: f"{v:10.2f}"))
print("\nla alternativa preferida cambia con los precios, de modo que el "
      "modelo hidrológico por sí solo no puede decidir")
print("lo que el modelo entrega es la frontera y el precio de cada mejora, "
      "no la respuesta")

Queda una distinción que el Capítulo 4 desarrolla y que conviene dejar
planteada. El cierre del balance de masa de la cascada de lagunas con
precisión de máquina y el orden dos observado en el método de líneas
prueban que el cálculo se hizo bien, no que el modelo represente el
sistema. Lo primero es verificación y lo segundo es validación, y ninguna
cantidad de la primera sustituye a la segunda.

## 6. Ficha de verificaciones de la Unidad 3

### Ejercicio 5

Complete la función que consolida las verificaciones de la unidad. Para
cada registro debe calcular la diferencia absoluta frente a la cifra del
libro y marcar si queda dentro de la tolerancia declarada. La celda de
partida marca todo como no verificado.

In [ ]:
# COMPLETE: agregue las columnas diferencia y dentro de tolerancia.
REVISAR_FICHA_FINAL = False


def consolidar(registros: list) -> pd.DataFrame:
    """Tabla de verificaciones frente a las cifras publicadas."""
    tabla = pd.DataFrame(registros, columns=["concepto", "calculado",
                                             "libro", "tolerancia"])
    tabla["diferencia"] = np.nan
    tabla["coincide"] = False
    return tabla

In [ ]:
from scipy.integrate import solve_ivp
from scipy.optimize import brentq, root
from scipy.sparse import diags

# cuaderno 1, régimen estacionario
Dtub, epstub, Qtub, nutub = 0.30, 0.26e-3, 0.12, 1.004e-6
Vtub = Qtub / (np.pi * Dtub**2 / 4)
Retub, rugtub = Vtub * Dtub / nutub, epstub / Dtub
residuo = lambda f: 1 / np.sqrt(f) + 2 * np.log10(rugtub / 3.7
                                                  + 2.51 / (Retub * np.sqrt(f)))
f_darcy = brentq(residuo, 0.005, 0.10, xtol=1e-14)

par = dict(Q=1200.0, Qr=600.0, V=2000.0, S0=250.0, kmax=60.0, Ks=40.0)

def residuos_cascada(S):
    r = par["kmax"] * S / (par["Ks"] + S)
    R = np.empty_like(S)
    R[0] = par["Q"] * par["S0"] - (par["Q"] + par["Qr"]) * S[0] \
        + par["Qr"] * S[1] - par["V"] * r[0]
    R[1:-1] = ((par["Q"] + par["Qr"]) * S[:-2]
               - (par["Q"] + 2 * par["Qr"]) * S[1:-1]
               + par["Qr"] * S[2:] - par["V"] * r[1:-1])
    R[-1] = (par["Q"] + par["Qr"]) * S[-2] \
        - (par["Q"] + par["Qr"]) * S[-1] - par["V"] * r[-1]
    return R

perfil = root(residuos_cascada, np.full(5, 250.0), tol=1e-12).x

# cuaderno 2, régimen dinámico
def fermentador(t, y, mu):
    X, S = y
    v = mu * S / (1.2 + S)
    return [v * X, -v * X / 0.08]

def agotamiento(t, y, mu):
    return y[1] - 1.0

agotamiento.direction = -1
agotamiento.terminal = True
lote = solve_ivp(fermentador, (0.0, 40.0), [0.50, 150.0], args=(0.30,),
                 method="RK45", rtol=1e-9, atol=1e-11, events=agotamiento)

# cuaderno 2, régimen distribuido
RIO = dict(u=0.35, D=12.0, k=0.25 / 86400, M=25.0, area=18.0, x0=1200.0)
malla = np.linspace(0.0, 5000.0, 251)
x_int, dx = malla[1:-1], malla[1] - malla[0]
exacta = lambda x, t: (RIO["M"] / (RIO["area"] * np.sqrt(np.pi * 4 * RIO["D"] * t))
                       * np.exp(-(x - RIO["x0"] - RIO["u"] * t)**2
                                / (4 * RIO["D"] * t)) * np.exp(-RIO["k"] * t))
n_nodos = x_int.size
A = diags([np.full(n_nodos - 1, RIO["D"] / dx**2 + RIO["u"] / (2 * dx)),
           np.full(n_nodos, -2 * RIO["D"] / dx**2 - RIO["k"]),
           np.full(n_nodos - 1, RIO["D"] / dx**2 - RIO["u"] / (2 * dx))],
          [-1, 0, 1], format="csc")
pluma = solve_ivp(lambda t, c: A @ c, (600.0, 3600.0), exacta(x_int, 600.0),
                  method="BDF", jac=lambda t, c: A, rtol=1e-11, atol=1e-16,
                  t_eval=[3600.0]).y[:, 0]
masa = float(trapecio(pluma, x_int) * RIO["area"])
print("resultados recalculados para la ficha de cierre")

In [ ]:
registros = [
    ("factor de fricción de Colebrook", f_darcy,
     LIBRO["colebrook_factor_friccion"], 1e-6),
    ("perfil de la laguna 1", perfil[0], LIBRO["lagunas_perfil_1"], 5e-3),
    ("perfil de la laguna 5", perfil[4], LIBRO["lagunas_perfil_5"], 5e-3),
    ("remoción global de la cascada", 100 * (1 - perfil[-1] / 250.0),
     LIBRO["lagunas_remocion"], 5e-3),
    ("agotamiento del fermentador a 30 grados",
     float(lote.t_events[0][0]), LIBRO["fermentador_tiempo_30C"], 1e-4),
    ("evaluaciones del fermentador", float(lote.nfev),
     LIBRO["fermentador_evaluaciones"], 0.0),
    ("masa remanente en el río", masa, LIBRO["rio_masa_remanente"], 1e-6),
    ("lámina bruta de la alternativa 0.45 y 45 mm",
     float(normal.query("p == 0.45 and d == 45.0")["bruto"].iloc[0]),
     LIBRO["riego_frontera_bruto_p045_d45"], 0.05),
    ("alternativas no dominadas en clima normal", float(len(frente)),
     LIBRO["riego_no_dominadas_clima_normal"], 0.0),
    ("media de la energía del biodigestor", float(E.mean()),
     LIBRO["biogas_media"], 5e-3),
    ("percentil cinco de la energía", float(np.percentile(E, 5)),
     LIBRO["biogas_p05"], 5e-3),
    ("probabilidad de cubrir la demanda", float((E >= DEMANDA).mean()),
     LIBRO["biogas_excedencia_110"], 5e-5),
]
ficha_final = consolidar(registros)
print(ficha_final.to_string(index=False, float_format=lambda v: f"{v:14.6g}"))
print(f"\nverificaciones que coinciden   {int(ficha_final['coincide'].sum())} "
      f"de {len(ficha_final)}")

if REVISAR_FICHA_FINAL:
    assert bool(ficha_final["coincide"].all()), \
        "alguna verificación de la unidad no coincide con el libro"
    print("la unidad completa queda verificada contra las cifras publicadas")
else:
    print("complete la celda anterior y ponga REVISAR_FICHA_FINAL = True")

## 7. Problemas de diseño y ensayo

El problema 3-29 pide diseñar el estudio que permita recomendar el número
de lagunas en serie de una planta para 18000 habitantes con volumen total
fijo. El andamiaje siguiente declara factores, niveles e indicadores, y
resuelve el barrido con el modelo de la cascada del cuaderno primero. La
recomendación se sustenta con la frontera y con el precio de cada mejora,
tal como se hizo con el riego.

In [ ]:
DOTACION, RETORNO, CARGA_PER_CAPITA = 150.0, 0.80, 45.0   # L/(hab d), -, g/(hab d)
HABITANTES, VOLUMEN_TOTAL = 18000, 30000.0                 # habitantes y m3
KMAX, KS = 60.0, 40.0                                      # mg/(L d) y mg/L
CAUDAL = HABITANTES * DOTACION * RETORNO / 1000.0           # m3/d
DBO_AFLUENTE = HABITANTES * CARGA_PER_CAPITA / CAUDAL       # mg/L
print(f"caudal de diseño            {CAUDAL:,.0f} m3/d")
print(f"DBO afluente                {DBO_AFLUENTE:,.1f} mg/L")
print(f"tiempo de retención total   {VOLUMEN_TOTAL / CAUDAL:,.2f} d")


def piezas_cascada(n: int, Qr_relativo: float):
    """Residuo y jacobiano de una cascada de n lagunas iguales."""
    V, Qr = VOLUMEN_TOTAL / n, Qr_relativo * CAUDAL
    Q, S0 = CAUDAL, DBO_AFLUENTE

    def residuo(S):
        r = KMAX * S / (KS + S)
        if n == 1:
            return np.array([Q * S0 - Q * S[0] - V * r[0]])
        R = np.empty_like(S)
        R[0] = Q * S0 - (Q + Qr) * S[0] + Qr * S[1] - V * r[0]
        R[1:-1] = ((Q + Qr) * S[:-2] - (Q + 2 * Qr) * S[1:-1]
                   + Qr * S[2:] - V * r[1:-1])
        R[-1] = (Q + Qr) * S[-2] - (Q + Qr) * S[-1] - V * r[-1]
        return R

    def jacobiano(S):
        dr = KMAX * KS / (KS + S)**2
        if n == 1:
            return np.array([[-Q - V * dr[0]]])
        J = np.diag(-(Q + 2 * Qr) - V * dr)
        J += np.diag(np.full(n - 1, Q + Qr), -1)
        J += np.diag(np.full(n - 1, Qr), 1)
        J[0, 0] = -(Q + Qr) - V * dr[0]
        J[-1, -1] = -(Q + Qr) - V * dr[-1]
        return J

    return residuo, jacobiano, V


def newton_amortiguado(F, J, x0, dominio, tol=1e-12, maxit=80,
                       lam_min=1e-6):
    """Newton con retroceso, según el Algoritmo 3.1 del libro."""
    x = np.array(x0, dtype=float)
    r = F(x)
    rho = np.max(np.abs(r))
    rho0 = rho
    for k in range(1, maxit + 1):
        dx = np.linalg.solve(J(x), -r)
        lam = 1.0
        while True:
            x_mas = x + lam * dx
            if dominio(x_mas):
                r_mas = F(x_mas)
                if np.max(np.abs(r_mas)) < rho:
                    break
            lam *= 0.5
            if lam < lam_min:
                return x, k, False
        x, r = x_mas, r_mas
        rho = np.max(np.abs(r))
        if rho <= tol * rho0:
            return x, k, True
    return x, maxit, False

Conviene ver primero por qué este barrido exige el Newton amortiguado. La
cinética de saturación tiene un polo en el valor opuesto de la constante
de semisaturación, de modo que el residuo posee raíces negativas sin
sentido físico. Partiendo del perfil uniforme, el algoritmo híbrido de
`scipy.optimize.root` puede declarar éxito sobre una de ellas, que es
exactamente la falla que anuncia la Tabla 3.1 del libro para Newton
cuando el punto inicial está lejos. El retroceso con predicado de dominio
del Algoritmo 3.1 evita esa trampa.

In [ ]:
from scipy.optimize import root

residuo_8, jacobiano_8, _ = piezas_cascada(8, 0.0)
ingenuo = root(residuo_8, np.full(8, DBO_AFLUENTE), tol=1e-12)
print("root desde el perfil uniforme, éxito declarado", ingenuo.success)
print("perfil obtenido", np.round(ingenuo.x, 2))
negativas = bool(np.any(ingenuo.x < 0))
print("hay concentraciones negativas", negativas)
if negativas:
    print("el algoritmo declara éxito sobre una raíz sin sentido físico, "
          "de modo que la bandera de convergencia no basta como criterio")

In [ ]:
dominio_fisico = lambda S: bool(np.all(S > 1e-9))
registros_diseno = []
for n in (1, 2, 3, 4, 5, 6, 8):
    for q in (0.0, 0.5, 1.0):
        F, J, V = piezas_cascada(n, q)
        partida = DBO_AFLUENTE * np.exp(-0.7 * np.arange(n))
        S, iteraciones, exito = newton_amortiguado(F, J, partida,
                                                   dominio_fisico)
        consumo = V * np.sum(KMAX * S / (KS + S))
        cierre = ((CAUDAL * DBO_AFLUENTE - CAUDAL * S[-1] - consumo)
                  / (CAUDAL * DBO_AFLUENTE))
        registros_diseno.append(dict(n=n, Qr_relativo=q,
                                     efluente=float(S[-1]),
                                     remocion=100 * (1 - S[-1] / DBO_AFLUENTE),
                                     iteraciones=iteraciones,
                                     cierre=float(cierre), convergio=exito))
estudio = pd.DataFrame(registros_diseno)
print(estudio.pivot_table(index="n", columns="Qr_relativo",
                          values="remocion").to_string(
    float_format=lambda v: f"{v:8.2f}"))
print(f"\ncorridas que no convergieron   {int((~estudio['convergio']).sum())}")
print(f"cierre relativo máximo         {estudio['cierre'].abs().max():.2e}")
print(f"perfiles con valores negativos {int((estudio['efluente'] < 0).sum())}")
assert estudio["convergio"].all(), "ninguna corrida debe quedar sin converger"
assert estudio["cierre"].abs().max() < 1e-10, \
    "el balance global debe cerrar en todas las corridas"
assert (estudio["efluente"] > 0).all(), \
    "todas las soluciones deben respetar el dominio físico"

In [ ]:
fig, ax = plt.subplots(figsize=(8.0, 4.2))
for q, color in zip((0.0, 0.5, 1.0), (COLORES["azul"], COLORES["verde"],
                                      COLORES["naranja"])):
    sub = estudio.query("Qr_relativo == @q")
    ax.plot(sub["n"], sub["remocion"], "o-", color=color,
            label=f"retromezcla {q:.1f} veces el caudal")
ax.set_xlabel("número de lagunas en serie, con volumen total fijo")
ax.set_ylabel("remoción global de DBO (por ciento)")
ax.set_title("Estudio de diseño para 18000 habitantes")
ax.legend(fontsize=8.5)
plt.show()

mejor = estudio.query("Qr_relativo == 0.0").sort_values("remocion").iloc[-1]
ganancia = estudio.query("Qr_relativo == 0.0").set_index("n")["remocion"]
print("ganancia marginal de remoción al agregar una laguna, en puntos")
print(ganancia.diff().dropna().to_string(float_format=lambda v: f"{v:6.2f}"))
print(f"\nla remoción crece con el número de unidades y satura, de modo que "
      f"la recomendación defendible se apoya en la ganancia marginal y no "
      f"en el máximo, que siempre se alcanza con el mayor número ensayado")
print("la retromezcla degrada la remoción porque acerca la cascada a un "
      "reactor de mezcla completa único")

## 8. Cierre de la Unidad 3

Al terminar este cuaderno el estudiante debe poder hacer lo siguiente.

1. Leer sobre la frontera de compromiso el precio de cada mejora y
   expresarlo en las unidades del problema. Revise la sección 3.4.1 si la
   frontera queda vacía o completa.
2. Distinguir la mejor alternativa en un contexto de la alternativa
   robusta a través de contextos, y declarar cuál de las dos se
   recomienda. Revise la Definición 3.5.
3. Redactar el conjunto mínimo defendible que acompaña a una simulación
   estocástica. Revise el cierre de la sección 3.5 si el informe se queda
   en la media.
4. Sustentar con cifras por qué el percentil bajo y no la media es la
   magnitud de diseño cuando el suministro debe garantizarse. Revise el
   comentario del Ejemplo 3.6.
5. Declarar qué información falta para decidir y qué distingue la
   verificación del cálculo de la validación del modelo, asunto que
   desarrolla el Capítulo 4.

Con esto se cierra la Unidad 3. Los resultados ya están producidos y
decidir cuánto de ellos es creíble corresponde a la Unidad 4.